In [50]:
import io
import zipfile
import requests
import frontmatter
from ingest_chunk import read_repo_data
from dotenv import load_dotenv
load_dotenv()

True

In [51]:
url = 'https://codeload.github.com/DataTalksClub/faq/zip/refs/heads/main'
resp = requests.get(url)

In [52]:
repository_data = []

# Create a ZipFile object from the downloaded content
zf = zipfile.ZipFile(io.BytesIO(resp.content))

for file_info in zf.infolist():
    filename = file_info.filename.lower()

    # Only process markdown files
    if not filename.endswith('.md'):
        continue

    # Read and parse each file
    with zf.open(file_info) as f_in:
        content = f_in.read()
        post = frontmatter.loads(content)
        data = post.to_dict()
        data['filename'] = filename
        repository_data.append(data)

zf.close()

In [53]:
print(repository_data[1])

{'name': 'slack-faq-fetch', 'description': "Pull recent Slack channel discussions for a course (e.g. LLM Zoomcamp, Data Engineering Zoomcamp) into a review export to find missing FAQ entries. Determines the lookback window from the run log / git history, runs the fetcher, logs the run, and points to the export for review. Use when the user wants to fetch Slack questions, refresh FAQ content from Slack, or check what's missing for a course.", 'content': '# Slack FAQ Fetch\n\nFetch recent Slack discussions for a course so missing FAQ entries can be curated. The fetcher is an existing Python CLI (`faq_automation/slack_fetch.py`); this skill drives it end to end.\n\n## How it works\n\n`faq_automation.slack_fetch` calls the Slack Web API using the `SLACK_BOT_TOKEN` from `.env` (loaded automatically — **never print the token**), reads the course\'s `slack_channel` from `_questions/<course>/_metadata.yaml`, pulls the last `N` days of messages **plus thread replies**, and writes paired JSON + 

In [54]:
for file_info in zf.infolist():
    filename = file_info.filename.lower()

    if not (filename.endswith('.md') or filename.endswith('.mdx')):
        continue

    # rest remains the same...

In [55]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
evidently_docs = read_repo_data('evidentlyai', 'docs')

print(f"FAQ documents: {len(dtc_faq)}")
print(f"Evidently documents: {len(evidently_docs)}")

FAQ documents: 1382
Evidently documents: 95


In [56]:
from ingest_chunk import chunk_documents

In [57]:
chunked_dtc_faq = chunk_documents(dtc_faq)

# |search

In [58]:
from minsearch import Index
de_dtc_faq = [d for d in dtc_faq if 'data-engineering' in d['filename']]


faq_index = Index(
    text_fields=["question", "content"],
    keyword_fields=[]
)

faq_index.fit(de_dtc_faq)

In [59]:
query = 'What should be in a test dataset for AI evaluation?'
results = faq_index.search(query)

In [60]:
results

[{'id': '9f9a1b9e4f',
  'question': 'Terraform: Teardown of BigQuery Dataset',
  'sort_order': 13,
  'content': "When running `terraform destroy`, the following error can occur:\n\n```\nDo you really want to destroy all resources?\n\nTerraform will destroy all your managed infrastructure, as shown above.\n\nThere is no undo. Only 'yes' will be accepted to confirm.\n\nEnter a value: yes\n\ngoogle_bigquery_dataset.homework_dataset: Destroying... [id=projects/terraform-demo-449214/datasets/homework_dataset]\n\n╷\n\n│ Error: Error when reading or editing Dataset: googleapi: Error 400: Dataset terraform-demo-449214:homework_dataset is still in use, resourceInUse\n```\n\nThis is because the dataset is still in use by a table. To delete the dataset, set the `delete_contents_on_destroy` property to `true` in the `main.tf` file.",
  'filename': 'faq-main/_questions/data-engineering-zoomcamp/module-1-terraform/013_9f9a1b9e4f_terraform-teardown-of-bigquery-dataset.md'},
 {'id': '8bfd724e4f',
  'q

# embedding_model

In [61]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [62]:
record = de_dtc_faq[2]
text = record['question'] + ' ' + record['content']
v_doc = embedding_model.encode(text)

In [63]:
query = 'I just found out about the course. Can I enroll now?'
v_query = embedding_model.encode(query)

In [64]:
from tqdm.auto import tqdm
import numpy as np

faq_embeddings = []

for d in tqdm(de_dtc_faq):
    text = d['question'] + ' ' + d['content']
    v = embedding_model.encode(text)
    faq_embeddings.append(v)

faq_embeddings = np.array(faq_embeddings)

  0%|          | 0/404 [00:00<?, ?it/s]

# VectorSearch

In [65]:
from minsearch import VectorSearch

faq_vindex = VectorSearch()
faq_vindex.fit(faq_embeddings, de_dtc_faq)

In [66]:
query = 'Can I join the course now?'
q = embedding_model.encode(query)
results = faq_vindex.search(q)

In [67]:
query = 'Can I join the course now?'

text_results = faq_index.search(query, num_results=5)

q = embedding_model.encode(query)
vector_results = faq_vindex.search(q, num_results=5)

final_results = text_results + vector_results

In [68]:
def text_search(query):
    return faq_index.search(query, num_results=5)

def vector_search(query):
    q = embedding_model.encode(query)
    return faq_vindex.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)

    # Combine and deduplicate results
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result['filename'] not in seen_ids:
            seen_ids.add(result['filename'])
            combined_results.append(result)

    return combined_results

In [69]:
import openai
import os

# client&tool 

In [70]:
openai_client = openai.OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

user_prompt = "I just discovered the course, can I join now?"

chat_messages = [
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='openai/gpt-oss-120b',
    input=chat_messages,
)

print(response.output_text)

Absolutely—you can still enroll! Just make sure you sign up before the deadline (July 31, 2024) and meet the prerequisites (e.g., the required background knowledge or any prerequisite courses listed). If you haven’t already, head to the enrollment page, fill out the form, and you’ll be all set to start the course. Let me know if you need help with any part of the registration process!


In [71]:
def text_search(query):
    return faq_index.search(query, num_results=5)

In [72]:
text_search_tool = {
    "type": "function",
    "name": "text_search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [73]:
system_prompt = """
You are a helpful assistant for a course.
"""

question = "I just discovered the course, can I join now?"

chat_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='openai/gpt-oss-120b',
    input=chat_messages,
    tools=[text_search_tool]
)

In [74]:
response.output

[ResponseReasoningItem(id='resp_01kxgb23anf769r4ge2hmxkf95', summary=[], type='reasoning', content=[Content(text='User asks: "I just discovered the course, can I join now?" Likely need to answer about enrollment periods, prerequisites. Could check FAQ. Use function text_search with query about joining the course.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course now? enrollment"}', call_id='fc_21c2b1b0-9598-47a0-a436-a811c0f6bf7d', name='text_search', type='function_call', id='fc_21c2b1b0-9598-47a0-a436-a811c0f6bf7d', namespace=None, status='completed')]

In [75]:
import json

call = response.output[1]

arguments = json.loads(call.arguments)
result = text_search(**arguments)

call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": json.dumps(result),
}

In [76]:
chat_messages.append(call)
chat_messages.append(call_output)

response = openai_client.responses.create(
    model='openai/gpt-oss-120b',
    input=chat_messages,
    tools=[text_search_tool]
)

print(response.output_text)

Yes—you can still hop on the course even if the cohort has already started.

**What you can do right now**

| Step | What to do | Why it helps |
|------|------------|--------------|
| **Register** | Grab the registration link from the course repository’s README (or the Telegram/Slack announcements). | Gives you access to the private Slack/Telegram channels, the assignment repo, and the official mailing list. |
| **Check the cohort timeline** | A typical cohort runs roughly January – April. Look at the current cohort’s exact start date in the repo. | Lets you know when the next big “kick‑off” events (live sessions, office hours) happen. |
| **Start the homework** | Even if you missed the official start, you’re still allowed to submit the homework assignments (subject to their deadlines). | You’ll stay on track with the material and can still earn a certificate or showcase the projects. |
| **Set up your environment** | Follow the “What can I do before the course starts?” checklist: crea

In [77]:
system_prompt = """
You are a helpful assistant for a course.

Use the search tool to find relevant information from the course materials before answering questions.

If you can find specific information through search, use it to provide accurate answers.
If the search doesn't return relevant results, let the user know and provide general guidance.
"""

In [78]:
system_prompt = """
You are a helpful assistant for a course.

Always search for relevant information before answering.
If the first search doesn't give you enough information, try different search terms.

Make multiple searches if needed to provide comprehensive answers.
"""

In [79]:
from typing import List, Any

def text_search(query: str) -> List[Any]:
    """
    Perform a text-based search on the FAQ index.

    Args:
        query (str): The search query string.

    Returns:
        List[Any]: A list of up to 5 search results returned by the FAQ index.
    """
    return faq_index.search(query, num_results=5)

In [80]:
from pydantic_ai.models.openai import OpenAIChatModel

In [81]:
import inspect
from pydantic_ai.models.openai import OpenAIChatModel

print(inspect.signature(OpenAIChatModel))

(model_name: 'OpenAIModelName', *, provider: "OpenAIChatCompatibleProvider | Literal['openai', 'openai-chat', 'gateway'] | Provider[AsyncOpenAI]" = 'openai', profile: 'ModelProfileSpec | None' = None, settings: 'ModelSettings | None' = None)


In [82]:
from pydantic_ai.providers.openai import OpenAIProvider

provider = OpenAIProvider(openai_client=openai_client)

In [83]:
model = OpenAIChatModel(
    model_name="openai/gpt-oss-120b",
    provider=provider,
)

In [84]:
from pydantic_ai import Agent



agent = Agent(
    "groq:openai/gpt-oss-120b",
    name="faq_agent",
    instructions=system_prompt,
    tools=[text_search],
)

In [85]:
question = "I just discovered the course, can I join now?"

result = await agent.run(user_prompt=question)

In [86]:
result.new_messages()

[ModelRequest(parts=[UserPromptPart(content='I just discovered the course, can I join now?', timestamp=datetime.datetime(2026, 7, 14, 12, 54, 5, 621715, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 7, 14, 12, 54, 5, 622178, tzinfo=datetime.timezone.utc), instructions="You are a helpful assistant for a course.\n\nAlways search for relevant information before answering.\nIf the first search doesn't give you enough information, try different search terms.\n\nMake multiple searches if needed to provide comprehensive answers.", run_id='019f60b1-1434-733a-868e-dba2c00946ec', conversation_id='019f60b1-1434-733a-868e-dba1c046f375'),
 ModelResponse(parts=[ThinkingPart(content='We need to answer user about joining the course now. Need to search FAQ.'), ToolCallPart(tool_name='text_search', args='{"query":"join course now enrollment open"}', tool_call_id='fc_90a3a819-a90c-445c-892e-98ea13842353')], usage=RequestUsage(input_tokens=228, output_tokens=48, details={'reasoning_to

In [87]:
from logs import log_interaction_to_file

In [88]:
question = input()
result = await agent.run(user_prompt=question)
print(result.output)
log_interaction_to_file(agent, result.new_messages())

Hello! How can I help you today?


PosixPath('logs/faq_agent_20260714_125420_143dcf.json')

In [89]:
system_prompt = """
You are a helpful assistant for a course.

Use the search tool to find relevant information from the course materials before answering questions.

If you can find specific information through search, use it to provide accurate answers.

Always include references by citing the filename of the source material you used.
When citing the reference, replace "faq-main" by the full path to the GitHub repository: "https://github.com/DataTalksClub/faq/blob/main/"
Format: [LINK TITLE](FULL_GITHUB_LINK)

If the search doesn't return relevant results, let the user know and provide general guidance.
"""

# Create another version of agent, let's call it faq_agent_v2
agent = Agent(
    name="faq_agent_v2",
    instructions=system_prompt,
    tools=[text_search],
    model="groq:openai/gpt-oss-120b"
)

In [90]:
evaluation_prompt = """
Use this checklist to evaluate the quality of an AI agent's answer (<ANSWER>) to a user question (<QUESTION>).
We also include the entire log (<LOG>) for analysis.

For each item, check if the condition is met.

Checklist:

- instructions_follow: The agent followed the user's instructions (in <INSTRUCTIONS>)
- instructions_avoid: The agent avoided doing things it was told not to do
- answer_relevant: The response directly addresses the user's question
- answer_clear: The answer is clear and correct
- answer_citations: The response includes proper citations or sources when required
- completeness: The response is complete and covers all key aspects of the request
- tool_call_search: Is the search tool invoked?

Output true/false for each check and provide a short explanation for your judgment.
""".strip()

In [91]:
from pydantic import BaseModel

class EvaluationCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool

class EvaluationChecklist(BaseModel):
    checklist: list[EvaluationCheck]
    summary: str

In [113]:
eval_agent = Agent(
    name='eval_agent',
    model="groq:openai/gpt-oss-120b",
    instructions=evaluation_prompt,
    output_type=EvaluationChecklist
)

In [93]:
user_prompt_format = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()

In [94]:
def load_log_file(log_file):
    with open(log_file, 'r') as f_in:
        log_data = json.load(f_in)
        log_data['log_file'] = log_file
        return log_data

In [96]:
log_record = load_log_file('/home/sam/Desktop/7Day-AI-Agents/aihero/course/logs/faq_agent_20260714_082429_a26152.json')


instructions = log_record['system_prompt']
question = log_record['messages'][0]['parts'][0]['content']
answer = log_record['messages'][-1]['parts'][0]['content']
log = json.dumps(log_record['messages'])


user_prompt = user_prompt_format.format(
    instructions=instructions,
    question=question,
    answer=answer,
    log=log
)

In [97]:
result = await eval_agent.run(user_prompt, output_type=EvaluationChecklist)

checklist = result.output

print(checklist.summary)

for check in checklist.checklist:
    print(check)

The assistant gave a clear, relevant greeting but failed to follow the mandatory search instruction, resulting in false for both instructions_follow and tool_call_search.
check_name='instructions_follow' justification='The user instructions required always performing a search before answering, but the assistant replied without any search.' check_pass=False
check_name='instructions_avoid' justification='The response contains no prohibited content and respects all avoidance constraints.' check_pass=True
check_name='answer_relevant' justification="The assistant answered the user's greeting appropriately." check_pass=True
check_name='answer_clear' justification='The greeting is straightforward and easy to understand.' check_pass=True
check_name='answer_citations' justification='No citations are required for a simple greeting.' check_pass=True
check_name='completeness' justification='The reply fully satisfies the conversational need for a greeting.' check_pass=True
check_name='tool_call_sea

In [98]:
def simplify_log_messages(messages):
    log_simplified = []

    for m in messages:
        parts = []

        for original_part in m['parts']:
            part = original_part.copy()
            kind = part['part_kind']

            if kind == 'user-prompt':
                del part['timestamp']
            if kind == 'tool-call':
                del part['tool_call_id']
            if kind == 'tool-return':
                del part['tool_call_id']
                del part['metadata']
                del part['timestamp']
                # Replace actual search results with placeholder to save tokens
                part['content'] = 'RETURN_RESULTS_REDACTED'
            if kind == 'text':
                del part['id']

            parts.append(part)

        message = {
            'kind': m['kind'],
            'parts': parts
        }

        log_simplified.append(message)
    return log_simplified

In [99]:
async def evaluate_log_record(eval_agent, log_record):
    messages = log_record['messages']

    instructions = log_record['system_prompt']
    question = messages[0]['parts'][0]['content']
    answer = messages[-1]['parts'][0]['content']

    log_simplified = simplify_log_messages(messages)
    log = json.dumps(log_simplified)

    user_prompt = user_prompt_format.format(
        instructions=instructions,
        question=question,
        answer=answer,
        log=log
    )

    result = await eval_agent.run(user_prompt, output_type=EvaluationChecklist)
    return result.output


log_record = load_log_file('/home/sam/Desktop/7Day-AI-Agents/aihero/course/logs/faq_agent_20260714_082429_a26152.json')
eval1 = await evaluate_log_record(eval_agent, log_record)

In [101]:
question_generation_prompt = """
You are helping to create test questions for an AI agent that answers questions about a data engineering course.

Based on the provided FAQ content, generate realistic questions that students might ask.

The questions should:

- Be natural and varied in style
- Range from simple to complex
- Include both specific technical questions and general course questions

Generate one question for each record.
""".strip()

class QuestionsList(BaseModel):
    questions: list[str]

question_generator = Agent(
    name="question_generator",
    instructions=question_generation_prompt,
    model='groq:openai/gpt-oss-120b',
    output_type=QuestionsList
)

In [102]:
import random

sample = random.sample(de_dtc_faq, 10)
prompt_docs = [d['content'] for d in sample]
prompt = json.dumps(prompt_docs)

result = await question_generator.run(prompt)
questions = result.output.questions

In [103]:
from tqdm.auto import tqdm

for q in tqdm(questions):
    print(q)

    result = await agent.run(user_prompt=q)
    print(result.output)

    log_interaction_to_file(
        agent,
        result.new_messages(),
        source='ai-generated'
    )

    print()

  0%|          | 0/10 [00:00<?, ?it/s]

Why do I get a "column does not exist" error in my join query when I write the column name without quotes, and how should I properly quote column identifiers?
When you write a column name in a PostgreSQL JOIN (or any other statement) **without quotes**, PostgreSQL treats the identifier as *unquoted*. Unquoted identifiers are automatically folded to **lower‑case** and must follow the rules for ordinary identifiers (letters, digits, underscores, and they cannot start with a digit).  

If the actual column name in the table was created with mixed‑case, spaces, or any other special character, the unquoted reference will not match the stored identifier, and PostgreSQL will raise:

```
ERROR:  column "myColumn" does not exist
```

Even putting the name in *single quotes* (`'myColumn'`) won’t help—single quotes create a **string literal**, not an identifier, so PostgreSQL still looks for a column named exactly that string and fails.

### Proper way to quote column identifiers  

Use **double 

In [105]:
from pathlib import Path

eval_set = []

LOG_DIR = Path('/home/sam/Desktop/7Day-AI-Agents/aihero/course/logs')

eval_set = []

for log_file in LOG_DIR.glob('*.json'):
    if 'faq_agent_v2' not in log_file.name:
        continue

    log_record = load_log_file(log_file)
    if log_record.get('source') != 'ai-generated':
        continue

    eval_set.append(log_record)

In [114]:
rows = []

rows = []

eval_results = [
    (log_record, await evaluate_log_record(eval_agent, log_record))
    for log_record in eval_set
]

for log_record, eval_result in eval_results:
    messages = log_record['messages']

    row = {
        'file': Path(log_record['log_file']).name,
        'question': messages[0]['parts'][0]['content'],
        'answer': messages[-1]['parts'][0]['content'],
    }

    checks = {c.check_name: c.check_pass for c in eval_result.checklist}
    row.update(checks)

    rows.append(row)

Traceback (most recent call last):
  File "/home/sam/Desktop/7Day-AI-Agents/aihero/course/.venv/lib/python3.12/site-packages/pydantic_ai/models/groq.py", line 83, in _map_api_errors
    yield
  File "/home/sam/Desktop/7Day-AI-Agents/aihero/course/.venv/lib/python3.12/site-packages/pydantic_ai/models/groq.py", line 380, in _completions_create
    return await self.client.chat.completions.create(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/sam/Desktop/7Day-AI-Agents/aihero/course/.venv/lib/python3.12/site-packages/groq/resources/chat/completions.py", line 943, in create
    return await self._post(
           ^^^^^^^^^^^^^^^^^
  File "/home/sam/Desktop/7Day-AI-Agents/aihero/course/.venv/lib/python3.12/site-packages/groq/_base_client.py", line 1856, in post
    return await self.request(cast_to, opts, stream=stream, stream_cls=stream_cls)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/sam/Desktop/7Day-AI-Agents/aihe

In [109]:
import pandas as pd

df_evals = pd.DataFrame(rows)

In [110]:
df_evals.mean(numeric_only=True)

Series([], dtype: float64)

In [111]:
def evaluate_search_quality(search_function, test_queries):
    results = []

    for query, expected_docs in test_queries:
        search_results = search_function(query, num_results=5)

        # Calculate hit rate
        relevant_found = any(doc['filename'] in expected_docs for doc in search_results)

        # Calculate MRR
        for i, doc in enumerate(search_results):
            if doc['filename'] in expected_docs:
                mrr = 1 / (i + 1)
                break
        else:
            mrr = 0

        results.append({
            'query': query,
            'hit': relevant_found,
            'mrr': mrr
        })
    return results
